# Autonomous Driving Danger Detection System
## Multi-Camera Perception | nuScenes Dataset

| Module | Model | Paper | Year |
|---|---|---|---|
| Object Detection | **YOLO11x** | Ultralytics | 2024 |
| Depth Estimation | **Depth Anything V2 Metric Outdoor** | Yang et al., NeurIPS | 2024 |
| Multi-Object Tracking | **StrongSORT** (via BoxMOT) | Du et al., IEEE TMM | 2023 |
| Optical Flow | **RAFT** (fallback: Farneback) | Teed & Deng, ECCV | 2020 |
| Panorama Stitching | **OpenCV SIFT + Homography** | — | — |
| Risk Scoring | **TTC + Multi-factor** | — | — |
| BEV | **nuScenes HD Map + Model outputs** | nuScenes | — |


In [51]:
import json, re

notebook_path = "/content/drive/MyDrive/Colab Notebooks/danger_detection_v1.ipynb"
with open(notebook_path) as f:
    content = f.read()

# search in outputs too
tokens = re.findall(r'ghp_[A-Za-z0-9]+', content)
urls = re.findall(r'https://ghp_[^\s"\\]+', content)
print('Tokens:', tokens)
print('URLs:', urls)

Tokens: []
URLs: []


In [ ]:
TOKEN = 'توکن_جدیدت'
USERNAME = 'zahrarezaei96'
REPO = 'CV_danger-detection'

!git config --global user.email "zahra.rezaei.academic@gmail.com"
!git config --global user.name "zahrarezaei96"

# پاک کردن repo قدیمی
!rm -rf /content/repo

# clone مجدد
!git clone https://{TOKEN}@github.com/{USERNAME}/{REPO}.git /content/repo

In [ ]:
%cd /content/repo
!cp "/content/drive/MyDrive/Colab Notebooks/danger_detection_v1.ipynb" .
!git add danger_detection_v1.ipynb
!git commit -m "v1: YOLO11x + Depth Anything V2 | F1=8.96% | Precision=7.14% | Recall=12.03%"
!git push

In [ ]:
# ── CELL 1: Install dependencies ────────────────────────────────────────────
import subprocess, sys, os
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0: print('ERR:', r.stderr[-200:])
    return r.returncode == 0

run('pip install -q "numpy<2.0"')
run('pip install -q nuscenes-devkit==1.2.0')
run('pip install -q ultralytics==8.3.40')
run('pip install -q pyquaternion==0.9.9')
run('pip install -q "imageio[ffmpeg]"')
run('pip install -q tqdm')
run('pip install -q "transformers>=4.50.0" "regex>=2025.10.22"')
run('pip install -q boxmot')

if not os.path.exists('/content/RAFT'):
    run('git clone -q https://github.com/princeton-vl/RAFT.git /content/RAFT')

print('All installed. Now: Runtime → Restart Runtime, then run from Cell 2.')


All installed. Now: Runtime → Restart Runtime, then run from Cell 2.


In [ ]:
# ── CELL 2: Imports ──────────────────────────────────────────────────────────
import sys, os, math, warnings, tempfile
from collections import defaultdict, deque

import numpy as np
import cv2
import torch
import imageio
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from tqdm import tqdm

from ultralytics import YOLO
from nuscenes.nuscenes import NuScenes
from nuscenes.map_expansion.map_api import NuScenesMap
from pyquaternion import Quaternion
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

sys.path.insert(0, '/content/RAFT/core')
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}  |  torch: {torch.__version__}')


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Device: cuda  |  torch: 2.10.0+cu128


In [ ]:
# ── CELL 3: Load nuScenes + Map from Google Drive ────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

NUSCENES_ROOT = '/content/drive/MyDrive/datasets/nuscenes'
os.makedirs(NUSCENES_ROOT, exist_ok=True)

if not os.path.exists(f'{NUSCENES_ROOT}/v1.0-mini'):
    print('Extracting nuScenes mini...')
    os.system(f'tar -xf "{NUSCENES_ROOT}/v1.0-mini.tgz" -C {NUSCENES_ROOT}')

if not os.path.exists(f'{NUSCENES_ROOT}/maps/expansion'):
    print('Extracting map expansion...')
    os.makedirs(f'{NUSCENES_ROOT}/maps/expansion', exist_ok=True)
    os.system(f'unzip -q "{NUSCENES_ROOT}/nuScenes-map-expansion-v1.3.zip" -d /tmp/map_ext')
    os.system(f'find /tmp/map_ext -name "*.json" -exec cp {{}} {NUSCENES_ROOT}/maps/expansion/ ;')
    os.system(f'find /tmp/map_ext -name "*.png"  -exec cp {{}} {NUSCENES_ROOT}/maps/ ;')
    print('Maps:', os.listdir(f'{NUSCENES_ROOT}/maps/expansion'))

nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_ROOT, verbose=False)
print(f'nuScenes loaded: {len(nusc.scene)} scenes')

MAP_LOCS = ['singapore-onenorth','singapore-queenstown',
            'singapore-hollandvillage','boston-seaport']
nusc_map = selected_scene = None
for scene in nusc.scene:
    log = nusc.get('log', scene['log_token'])
    loc = log['location']
    if loc in MAP_LOCS:
        try:
            nusc_map = NuScenesMap(dataroot=NUSCENES_ROOT, map_name=loc)
            selected_scene = scene
            print(f'Scene: {scene["name"]}  |  {loc}')
            break
        except Exception as e:
            print(f'Map fail {loc}: {e}')

assert selected_scene is not None, 'No scene with map found.'

samples = []
tok = selected_scene['first_sample_token']
while tok:
    s = nusc.get('sample', tok)
    samples.append(s)
    tok = s['next']
print(f'Frames: {len(samples)}')

CAM_NAMES  = ['CAM_FRONT','CAM_FRONT_RIGHT','CAM_FRONT_LEFT',
              'CAM_BACK', 'CAM_BACK_LEFT',  'CAM_BACK_RIGHT']
FRONT_CAMS = ['CAM_FRONT_LEFT','CAM_FRONT','CAM_FRONT_RIGHT']
BACK_CAMS  = ['CAM_BACK_RIGHT','CAM_BACK','CAM_BACK_LEFT']


Mounted at /content/drive
nuScenes loaded: 10 scenes
Scene: scene-0061  |  singapore-onenorth
Frames: 39


In [ ]:
# ── CELL 4: Load YOLO11x (Ultralytics 2024) ──────────────────────────────────
yolo = YOLO('yolo11x.pt')
yolo.to(DEVICE)

DETECT_CLASSES = {
    0:'person', 1:'bicycle', 2:'car', 3:'motorcycle',
    5:'bus', 7:'truck', 15:'cat', 16:'dog', 17:'horse',
    18:'sheep', 19:'cow',
}
print('YOLO11x loaded.')


100%|██████████| 109M/109M [00:03<00:00, 34.2MB/s]


YOLO11x loaded.


In [ ]:
# ── CELL 5: Depth Anything V2 Metric Outdoor (NeurIPS 2024) ──────────────────
print('Loading Depth Anything V2...')
da2_processor = AutoImageProcessor.from_pretrained(
    'depth-anything/Depth-Anything-V2-Metric-Outdoor-Large-hf')
da2_model = AutoModelForDepthEstimation.from_pretrained(
    'depth-anything/Depth-Anything-V2-Metric-Outdoor-Large-hf').to(DEVICE)
da2_model.eval()
print('Depth Anything V2 loaded.')


Loading Depth Anything V2...


preprocessor_config.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/503 [00:00<?, ?it/s]

Depth Anything V2 loaded.


In [ ]:
# ── CELL 6: StrongSORT tracker (BoxMOT, Du et al. IEEE TMM 2023) ─────────────
from boxmot import Boxmot
from pathlib import Path

trackers = {}
for cam in FRONT_CAMS:
    trackers[cam] = Boxmot(
        tracker='strongsort',
        reid=Path('osnet_x0_25_msmt17.pt'),
    )
print('StrongSORT trackers initialized.')

StrongSORT trackers initialized.


In [ ]:
# ── CELL 7: RAFT Optical Flow (Teed & Deng, ECCV 2020) ───────────────────────
import sys
sys.path.insert(0, '/content/RAFT/core')

try:
    from raft import RAFT
    from utils.utils import InputPadder
    import argparse
    args = argparse.Namespace(small=False, mixed_precision=False, alternate_corr=False)
    if not os.path.exists('/content/models/raft-things.pth'):
        os.system('bash /content/RAFT/download_models.sh')
    raft_model = RAFT(args)
    raft_model = torch.nn.DataParallel(raft_model)
    raft_model.load_state_dict(torch.load('/content/models/raft-things.pth', map_location=DEVICE))
    raft_model = raft_model.module.to(DEVICE).eval()
    USE_RAFT = True
    print('RAFT loaded.')
except Exception as e:
    print(f'RAFT load failed: {e} — Falling back to Farneback.')
    USE_RAFT = False


RAFT loaded.


In [ ]:
# ── CELL 8: nuScenes helper functions ────────────────────────────────────────

def get_ego_pose(sample):
    sd  = nusc.get('sample_data', sample['data']['CAM_FRONT'])
    ep  = nusc.get('ego_pose', sd['ego_pose_token'])
    x, y = ep['translation'][0], ep['translation'][1]
    q    = ep['rotation']
    yaw  = math.atan2(2*(q[0]*q[3]+q[1]*q[2]), 1-2*(q[2]**2+q[3]**2))
    return float(x), float(y), float(yaw)

def get_cam_image(sample, cam):
    sd  = nusc.get('sample_data', sample['data'][cam])
    img = cv2.imread(os.path.join(NUSCENES_ROOT, sd['filename']))
    return img

def get_intrinsics(sample, cam):
    sd = nusc.get('sample_data', sample['data'][cam])
    cs = nusc.get('calibrated_sensor', sd['calibrated_sensor_token'])
    return np.array(cs['camera_intrinsic'], dtype=np.float64)

def get_cam2ego(sample, cam):
    sd  = nusc.get('sample_data', sample['data'][cam])
    cs  = nusc.get('calibrated_sensor', sd['calibrated_sensor_token'])
    R   = Quaternion(cs['rotation']).rotation_matrix
    t   = np.array(cs['translation'])
    T   = np.eye(4, dtype=np.float64)
    T[:3,:3] = R; T[:3,3] = t
    return T

print('nuScenes helpers defined.')


nuScenes helpers defined.


In [ ]:
# ── CELL 9: Depth estimation helpers ─────────────────────────────────────────

def run_depth(img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = PILImage.fromarray(img_rgb)
    inputs  = da2_processor(images=pil_img, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = da2_model(**inputs)
    depth = da2_processor.post_process_depth_estimation(
        outputs,
        target_sizes=[(img_bgr.shape[0], img_bgr.shape[1])]
    )[0]['predicted_depth']
    return depth.squeeze().float().cpu().numpy()

def bbox_depth(depth_map, bbox):
    h, w = depth_map.shape
    x1 = max(0, int(bbox[0])); y1 = max(0, int(bbox[1]))
    x2 = min(w, int(bbox[2])); y2 = min(h, int(bbox[3]))
    patch = depth_map[y1:y2, x1:x2]
    return float(np.median(patch)) if patch.size > 0 else float('inf')

def estimate_depth_from_bbox(bbox, img_w, img_h):
    """Fallback depth estimate from bbox size — used for cameras without a depth map"""
    bw = bbox[2] - bbox[0]; bh = bbox[3] - bbox[1]
    area_ratio = (bw * bh) / max(img_w * img_h, 1)
    return float(np.clip(2.0 / max(area_ratio, 0.001), 1.0, 60.0))

print('Depth helpers defined.')


Depth helpers defined.


In [ ]:
# ── CELL 10: Optical flow velocity estimator ─────────────────────────────────

prev_frames = {}

def get_flow(cam, img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    flow = None
    if cam in prev_frames:
        if USE_RAFT:
            try:
                from utils.utils import InputPadder
                def to_tensor(g):
                    rgb = cv2.cvtColor(g, cv2.COLOR_GRAY2RGB)
                    return torch.from_numpy(rgb).permute(2,0,1).float()[None].to(DEVICE)
                img1 = to_tensor(prev_frames[cam])
                img2 = to_tensor(gray)
                padder = InputPadder(img1.shape)
                img1, img2 = padder.pad(img1, img2)
                with torch.no_grad():
                    _, flow_up = raft_model(img1, img2, iters=12, test_mode=True)
                flow = flow_up[0].permute(1,2,0).cpu().numpy()
            except Exception:
                flow = cv2.calcOpticalFlowFarneback(
                    prev_frames[cam], gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        else:
            flow = cv2.calcOpticalFlowFarneback(
                prev_frames[cam], gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    prev_frames[cam] = gray
    return flow

def bbox_flow(flow, bbox):
    if flow is None: return 0.0, 0.0, 0.0, 0.0
    x1,y1,x2,y2 = int(bbox[0]),int(bbox[1]),int(bbox[2]),int(bbox[3])
    h, w = flow.shape[:2]
    x1=max(0,x1); y1=max(0,y1); x2=min(w,x2); y2=min(h,y2)
    patch = flow[y1:y2, x1:x2]
    if patch.size == 0: return 0.0, 0.0, 0.0, 0.0
    vx = float(np.median(patch[...,0]))
    vy = float(np.median(patch[...,1]))
    return vx, vy, math.hypot(vx,vy), math.atan2(vy,vx)

print('Optical flow helpers defined.')


Optical flow helpers defined.


In [ ]:
# ── CELL 11: Risk / Danger scoring ──────────────────────────────────────────

CLASS_W = {
    'person':1.5, 'bicycle':1.3, 'motorcycle':1.3,
    'car':1.0, 'truck':1.0, 'bus':1.0,
    'horse':1.0, 'cow':0.8, 'dog':0.9,
}

RISK_COLOR_BGR = {
    'HIGH':   (50,  50,  220),
    'MEDIUM': (0,  140,  255),
    'LOW':    (0,  190,  190),
    'SAFE':   (80, 160,   80),
}

def assess_risk_ttc(depth_m, speed_px=0.0, class_name='car'):
    cw = CLASS_W.get(class_name, 1.0)
    if depth_m < 5:    prox = 1.0
    elif depth_m < 15: prox = 0.7
    elif depth_m < 30: prox = 0.35
    else:              prox = 0.1
    spd_s = min(1.0, speed_px / 10.0)
    ttc   = depth_m / max(speed_px * 0.5, 0.1)
    ttc_s = max(0.0, 1.0 - ttc/10.0)
    score = min(1.0, cw * (0.5*prox + 0.25*spd_s + 0.25*ttc_s))
    if   depth_m < 5:  level = 'HIGH'
    elif depth_m < 15: level = 'MEDIUM'
    elif depth_m < 30: level = 'LOW'
    else:              level = 'SAFE'
    return score, level

print('Risk scorer defined.')


Risk scorer defined.


In [ ]:
# ── CELL 12: World-space NMS — Remove duplicate detections across cameras ─

def nms_world(detections, threshold=3.0):
    """Keep the closest detection if two are within threshold meters in world space"""
    if not detections: return []
    kept = []
    used = set()
    dets = sorted(detections, key=lambda x: x['depth_model'])
    for i, d1 in enumerate(dets):
        if i in used: continue
        kept.append(d1)
        for j, d2 in enumerate(dets):
            if j <= i or j in used: continue
            dist = math.hypot(d1['wx']-d2['wx'], d1['wy']-d2['wy'])
            if dist < threshold:
                used.add(j)
    return kept

print('World NMS defined.')


World NMS defined.


In [ ]:
# ── CELL 13: Panorama stitcher (SIFT + Homography) ──────────────────────────
# Stitches three front/back cameras into a single seamless panorama image

def stitch_panorama(imgs):
    """
    imgs: list of 3 BGR images [left, center, right]
    returns: panorama BGR image
    """
    if len(imgs) != 3 or any(img is None for img in imgs):
        # fallback: stack images side by side
        valid = [img for img in imgs if img is not None]
        if not valid: return np.zeros((360, 1920, 3), np.uint8)
        h = min(img.shape[0] for img in valid)
        resized = [cv2.resize(img, (640, h)) for img in valid]
        return np.hstack(resized)

    try:
        sift = cv2.SIFT_create(nfeatures=2000)
        bf   = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)

        def match_and_stitch(img_left, img_right):
            gray_l = cv2.cvtColor(img_left,  cv2.COLOR_BGR2GRAY)
            gray_r = cv2.cvtColor(img_right, cv2.COLOR_BGR2GRAY)
            kp_l, des_l = sift.detectAndCompute(gray_l, None)
            kp_r, des_r = sift.detectAndCompute(gray_r, None)
            if des_l is None or des_r is None or len(kp_l)<4 or len(kp_r)<4:
                return np.hstack([img_left, img_right])
            matches = bf.knnMatch(des_r, des_l, k=2)
            good = [m for m,n in matches if m.distance < 0.75*n.distance]
            if len(good) < 8:
                return np.hstack([img_left, img_right])
            src_pts = np.float32([kp_r[m.queryIdx].pt for m in good]).reshape(-1,1,2)
            dst_pts = np.float32([kp_l[m.trainIdx].pt for m in good]).reshape(-1,1,2)
            H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
            if H is None:
                return np.hstack([img_left, img_right])
            h_l, w_l = img_left.shape[:2]
            h_r, w_r = img_right.shape[:2]
            w_out = w_l + w_r
            warped = cv2.warpPerspective(img_right, H, (w_out, h_l))
            result = warped.copy()
            result[:h_l, :w_l] = img_left
            # blend overlapping region between adjacent images
            overlap_x = w_l - int(w_r * 0.3)
            for x in range(max(0, overlap_x), min(w_l, w_out)):
                alpha = (x - overlap_x) / max(w_l - overlap_x, 1)
                alpha = np.clip(alpha, 0, 1)
                if x < warped.shape[1] and np.any(warped[:h_l, x] > 0):
                    result[:h_l, x] = (
                        (1-alpha) * img_left[:h_l, x] +
                        alpha * warped[:h_l, x]
                    ).astype(np.uint8)
            return result

        # stitch left + center
        left_center = match_and_stitch(imgs[0], imgs[1])
        # stitch (left+center) + right
        panorama    = match_and_stitch(left_center, imgs[2])
        return panorama

    except Exception as e:
        print(f'Stitch failed: {e} — fallback to hstack')
        h = min(img.shape[0] for img in imgs)
        return np.hstack([cv2.resize(img,(640,h)) for img in imgs])

print('Panorama stitcher defined.')


Panorama stitcher defined.


In [ ]:
# ── CELL 14: BEV renderer ────────────────────────────────────────────────────
MAP_SIZE  = 600
MAP_RANGE = 30.0
SCALE     = MAP_SIZE / (MAP_RANGE * 2)
CENTER    = MAP_SIZE // 2

ANN_SIZE = {
    'vehicle.car':        (4.5, 2.0),
    'vehicle.truck':      (8.0, 2.5),
    'vehicle.bus':       (12.0, 2.8),
    'vehicle.bicycle':    (1.8, 0.8),
    'vehicle.motorcycle': (2.2, 0.9),
    'human.pedestrian':   (0.8, 0.8),
    'movable_object':     (1.0, 1.0),
}
SKIP_CATS = ('movable_object.barrier','movable_object.trafficcone',
             'movable_object.debris','static_object')

def get_ann_size(cat):
    for k,v in ANN_SIZE.items():
        if cat.startswith(k): return v
    return (1.5, 1.5)

def world_to_bev(wx, wy, ego_x, ego_y, ego_yaw):
    dx = wx - ego_x; dy = wy - ego_y
    cos_e = math.cos(-ego_yaw); sin_e = math.sin(-ego_yaw)
    lx =  sin_e*dx + cos_e*dy
    ly =  cos_e*dx - sin_e*dy
    return int(CENTER - lx*SCALE), int(CENTER - ly*SCALE)

def build_map_bg(ego_x, ego_y, ego_yaw):
    canvas = np.full((MAP_SIZE, MAP_SIZE, 3), 255, np.uint8)
    patch  = (ego_x-MAP_RANGE, ego_y-MAP_RANGE,
              ego_x+MAP_RANGE, ego_y+MAP_RANGE)
    for layer in ['road_segment','lane']:
        try: hits = nusc_map.get_records_in_patch(patch,[layer],'intersect')
        except: continue
        for tok in hits.get(layer,[]):
            rec = nusc_map.get(layer, tok)
            pt  = rec.get('polygon_token')
            if not pt: continue
            try: poly = nusc_map.extract_polygon(pt)
            except: continue
            pts = np.array(poly.exterior.coords)
            pxs = np.array([world_to_bev(x,y,ego_x,ego_y,ego_yaw) for x,y in pts], np.int32)
            cv2.fillPoly(canvas,[pxs],(210,210,210))
            cv2.polylines(canvas,[pxs],True,(180,180,180),1)
    return canvas

def _draw_ego_and_legend(canvas):
    ev_l=max(10,int(4.5*SCALE/2)); ev_w=max(6,int(2.0*SCALE/2))
    cv2.rectangle(canvas,(CENTER-ev_w,CENTER-ev_l),(CENTER+ev_w,CENTER+ev_l),(30,30,200),-1)
    cv2.rectangle(canvas,(CENTER-ev_w,CENTER-ev_l),(CENTER+ev_w,CENTER+ev_l),(10,10,140),2)
    tri=np.array([[CENTER,CENTER-ev_l-8],[CENTER-6,CENTER-ev_l+2],[CENTER+6,CENTER-ev_l+2]],np.int32)
    cv2.fillPoly(canvas,[tri],(30,30,200))
    cv2.putText(canvas,'EGO',(CENTER-12,CENTER+5),cv2.FONT_HERSHEY_SIMPLEX,0.38,(255,255,255),1)
    cv2.rectangle(canvas,(6,6),(195,110),(245,245,245),-1)
    cv2.rectangle(canvas,(6,6),(195,110),(180,180,180),1)
    for i,(lv,col) in enumerate([('HIGH   (<5m)',RISK_COLOR_BGR['HIGH']),
        ('MEDIUM (5-15m)',RISK_COLOR_BGR['MEDIUM']),
        ('LOW    (15-30m)',RISK_COLOR_BGR['LOW']),
        ('SAFE   (>30m)',RISK_COLOR_BGR['SAFE'])]):
        cy_=22+i*22
        cv2.circle(canvas,(20,cy_),6,col,-1)
        cv2.putText(canvas,lv,(32,cy_+4),cv2.FONT_HERSHEY_SIMPLEX,0.33,(30,30,30),1)
    bar=int(10*SCALE); bx=MAP_SIZE-20-bar; by=MAP_SIZE-20
    cv2.line(canvas,(bx,by),(bx+bar,by),(80,80,80),2)
    cv2.putText(canvas,'10m',(bx,by-5),cv2.FONT_HERSHEY_SIMPLEX,0.38,(80,80,80),1)

def draw_bev_gt(sample, ego_x, ego_y, ego_yaw, ego_hist, obj_hist, depth_map):
    """Render BEV map using nuScenes ground-truth annotations"""
    canvas = build_map_bg(ego_x, ego_y, ego_yaw)
    for r in [5,10,20,30]:
        r_px=int(r*SCALE)
        cv2.circle(canvas,(CENTER,CENTER),r_px,(210,210,210),1,cv2.LINE_AA)
        cv2.putText(canvas,f'{r}m',(CENTER+r_px+2,CENTER),cv2.FONT_HERSHEY_SIMPLEX,0.35,(160,160,160),1)
    if len(ego_hist)>=2:
        pts=[]
        for gx,gy in ego_hist:
            px,py=world_to_bev(gx,gy,ego_x,ego_y,ego_yaw)
            if 0<=px<MAP_SIZE and 0<=py<MAP_SIZE: pts.append((px,py))
        for i in range(1,len(pts)):
            cv2.line(canvas,pts[i-1],pts[i],(0,160,80),2,cv2.LINE_AA)
    for ann_token in sample['anns']:
        ann = nusc.get('sample_annotation', ann_token)
        cat = ann['category_name']
        if any(cat.startswith(s) for s in SKIP_CATS): continue
        pos = ann['translation']
        yaw = Quaternion(ann['rotation']).yaw_pitch_roll[0]
        l,w = get_ann_size(cat)
        dx  = pos[0]-ego_x; dy = pos[1]-ego_y
        dist= math.hypot(dx,dy)
        if dist > MAP_RANGE: continue
        d_risk = dist
        if depth_map is not None:
            cos_y=math.cos(-ego_yaw); sin_y=math.sin(-ego_yaw)
            lx_f=sin_y*dx+cos_y*dy; ly_f=cos_y*dx-sin_y*dy
            if ly_f>0 and abs(math.degrees(math.atan2(lx_f,ly_f)))<35:
                dh,dw=depth_map.shape
                angle_h=math.atan2(lx_f,ly_f)
                px_norm=angle_h/math.radians(35)
                px_d=int((px_norm+1)/2*dw); py_d=int(dh*0.6)
                px_d=np.clip(px_d,5,dw-5); py_d=np.clip(py_d,5,dh-5)
                patch=depth_map[py_d-4:py_d+4,px_d-4:px_d+4]
                if patch.size>0 and (patch>0).any():
                    val=float(np.median(patch[patch>0]))
                    if 0.5<val<dist*2: d_risk=val
        score,level = assess_risk_ttc(d_risk, 0.0, cat.split('.')[-1])
        color = RISK_COLOR_BGR[level]
        cx_px,cy_px = world_to_bev(pos[0],pos[1],ego_x,ego_y,ego_yaw)
        if not (0<=cx_px<MAP_SIZE and 0<=cy_px<MAP_SIZE): continue
        if ann_token in obj_hist and len(obj_hist[ann_token])>=2:
            th=list(obj_hist[ann_token])
            tpts=[world_to_bev(tx,ty,ego_x,ego_y,ego_yaw) for tx,ty in th]
            tpts=[p for p in tpts if 0<=p[0]<MAP_SIZE and 0<=p[1]<MAP_SIZE]
            for i in range(1,len(tpts)):
                if i%2==0: cv2.line(canvas,tpts[i-1],tpts[i],(150,100,200),1)
        if cat.startswith('human.pedestrian'):
            cv2.circle(canvas,(cx_px,cy_px),5,color,-1,cv2.LINE_AA)
            cv2.circle(canvas,(cx_px,cy_px),5,(30,30,30),1,cv2.LINE_AA)
        else:
            bev_angle_deg=-math.degrees(yaw-ego_yaw)
            lp=int(l*SCALE); wp=int(w*SCALE)
            box=cv2.boxPoints(((cx_px,cy_px),(wp,lp),bev_angle_deg))
            box=np.intp(box)
            ov=canvas.copy()
            cv2.fillPoly(ov,[box],color)
            cv2.addWeighted(ov,0.45,canvas,0.55,0,canvas)
            cv2.polylines(canvas,[box],True,color,2)
            angle_rad=math.radians(bev_angle_deg)
            fx=int(cx_px+(lp/2)*math.sin(angle_rad))
            fy=int(cy_px-(lp/2)*math.cos(angle_rad))
            cv2.line(canvas,(cx_px,cy_px),(fx,fy),(30,30,30),2)
        if level in ('HIGH','MEDIUM'):
            cv2.putText(canvas,f'{cat.split(".")[-1][:4]} {d_risk:.0f}m',
                        (cx_px+int(w*SCALE)+2,cy_px-3),cv2.FONT_HERSHEY_SIMPLEX,0.35,color,1)
    cv2.putText(canvas,'GT',(10,MAP_SIZE-10),cv2.FONT_HERSHEY_SIMPLEX,0.5,(100,200,100),1)
    _draw_ego_and_legend(canvas)
    return canvas

def draw_bev_model(ego_x, ego_y, ego_yaw, ego_hist, all_dets):
    """Render BEV map using model outputs (YOLO11 + Depth Anything V2)"""
    canvas = build_map_bg(ego_x, ego_y, ego_yaw)
    for r in [5,10,20,30]:
        r_px=int(r*SCALE)
        cv2.circle(canvas,(CENTER,CENTER),r_px,(200,200,200),1,cv2.LINE_AA)
        cv2.putText(canvas,f'{r}m',(CENTER+r_px+2,CENTER),cv2.FONT_HERSHEY_SIMPLEX,0.35,(160,160,160),1)
    if len(ego_hist)>=2:
        pts=[]
        for gx,gy in ego_hist:
            px,py=world_to_bev(gx,gy,ego_x,ego_y,ego_yaw)
            if 0<=px<MAP_SIZE and 0<=py<MAP_SIZE: pts.append((px,py))
        for i in range(1,len(pts)):
            cv2.line(canvas,pts[i-1],pts[i],(0,160,80),2,cv2.LINE_AA)
    for mo in all_dets:
        bx,by=world_to_bev(mo['wx'],mo['wy'],ego_x,ego_y,ego_yaw)
        if not (0<=bx<MAP_SIZE and 0<=by<MAP_SIZE): continue
        score,level=assess_risk_ttc(mo['depth_model'],0.0,mo['cname'])
        color=RISK_COLOR_BGR[level]
        if mo['cname']=='person':
            cv2.circle(canvas,(bx,by),5,color,-1,cv2.LINE_AA)
        else:
            lm,wm={'car':(4.5,2.0),'truck':(8.0,2.5),'bus':(12.0,2.8),
                   'motorcycle':(2.2,0.9),'bicycle':(1.8,0.8)}.get(mo['cname'],(3.0,1.5))
            lp=int(lm*SCALE); wp=int(wm*SCALE)
            bp=cv2.boxPoints(((bx,by),(wp,lp),0)); bp=np.intp(bp)
            ov=canvas.copy()
            cv2.fillPoly(ov,[bp],color)
            cv2.addWeighted(ov,0.45,canvas,0.55,0,canvas)
            cv2.polylines(canvas,[bp],True,(30,30,30),1)
        if level in ('HIGH','MEDIUM'):
            cv2.putText(canvas,f'{mo["cname"][:3]} {mo["depth_model"]:.1f}m',
                        (bx+8,by),cv2.FONT_HERSHEY_SIMPLEX,0.35,color,1)
    cv2.putText(canvas,'MODEL',(10,MAP_SIZE-10),cv2.FONT_HERSHEY_SIMPLEX,0.5,(100,100,200),1)
    _draw_ego_and_legend(canvas)
    return canvas

print('BEV renderers defined.')


BEV renderers defined.


In [ ]:
# ── CELL 15: Camera frame annotator ─────────────────────────────────────────

def annotate_camera(img_bgr, dets):
    out = img_bgr.copy()
    for det in dets:
        x1,y1,x2,y2 = [int(v) for v in det['bbox']]
        level  = det.get('level','SAFE')
        cname  = det.get('class_name','?')
        tid    = det.get('track_id',-1)
        depth  = det.get('depth',0.0)
        color  = RISK_COLOR_BGR.get(level,(80,160,80))
        cv2.rectangle(out,(x1,y1),(x2,y2),color,2)
        lines=[f'#{tid} {cname} [{level}]', f'dist:{depth:.1f}m']
        for k,line in enumerate(lines):
            tw,th=cv2.getTextSize(line,cv2.FONT_HERSHEY_SIMPLEX,0.44,1)[0]
            ty=y1-4-(len(lines)-1-k)*(th+4)
            if ty<th+4: ty=y2+(k+1)*(th+6)
            cv2.rectangle(out,(x1,ty-th-2),(x1+tw+3,ty+2),color,-1)
            cv2.putText(out,line,(x1+2,ty),
                        cv2.FONT_HERSHEY_SIMPLEX,0.44,(255,255,255),1,cv2.LINE_AA)
    return out

print('Camera annotator defined.')


Camera annotator defined.


In [ ]:
# ── CELL 16: MAIN PROCESSING LOOP ───────────────────────────────────────────
MAX_FRAMES   = min(20, len(samples))
FPS_OUT      = 1
VIDEO_OUTPUT = '/content/danger_detection_final.mp4'
TARGET_H     = 430
TARGET_W     = 640
PANO_W       = TARGET_W * 3   # 1920 — total panorama width
BEV_SQ       = TARGET_H * 3   # BEV canvas size (square)

ego_hist = deque(maxlen=40)
obj_hist = defaultdict(lambda: deque(maxlen=20))

writer = imageio.get_writer(
    VIDEO_OUTPUT, fps=FPS_OUT, codec='libx264', quality=7, macro_block_size=2)

print(f'Processing {MAX_FRAMES} frames...')

for f_idx in tqdm(range(MAX_FRAMES)):
    sample = samples[f_idx]
    ego_x, ego_y, ego_yaw = get_ego_pose(sample)
    ego_hist.append((ego_x, ego_y))

    for ann_token in sample['anns']:
        ann = nusc.get('sample_annotation', ann_token)
        obj_hist[ann_token].append(ann['translation'][:2])

    # ── Run Depth Anything V2 on all 6 cameras ───────────────────────────────────
    depth_maps = {}
    for cam in CAM_NAMES:
        img_c = get_cam_image(sample, cam)
        if img_c is not None:
            depth_maps[cam] = run_depth(img_c)

    depth_map = depth_maps.get('CAM_FRONT')

    # ── Collect detections from all 6 cameras and apply world-space NMS ────────────────────────────
    all_model_dets = []
    cam_dets = {}  # per-camera detections for frame annotation

    for cam in CAM_NAMES:
        img_c = get_cam_image(sample, cam)
        if img_c is None: continue
        dm      = depth_maps.get(cam)
        K_c     = get_intrinsics(sample, cam)
        T_c2e_c = get_cam2ego(sample, cam)
        H_c,W_c = img_c.shape[:2]
        dets_raw = []

        if cam in trackers:
            res = yolo(img_c, conf=0.15,
                       classes=list(DETECT_CLASSES.keys()), verbose=False)[0]
            if res.boxes is not None and len(res.boxes):
                boxes_np = res.boxes.xyxy.cpu().numpy()
                confs_np = res.boxes.conf.cpu().numpy()
                clss_np  = res.boxes.cls.cpu().numpy()
                dets_in  = np.column_stack([boxes_np,confs_np,clss_np])
                try:
                    tracks = trackers[cam].update(dets_in, img_c)
                    if tracks is not None and len(tracks):
                        for tr in tracks:
                            x1,y1,x2,y2,tid,conf,cls_id,_ = tr[:8]
                            cid  = int(cls_id)
                            cname = DETECT_CLASSES.get(cid,'object')
                            d = bbox_depth(dm,[x1,y1,x2,y2]) if dm is not None else estimate_depth_from_bbox([x1,y1,x2,y2],W_c,H_c)
                            if d<=0 or d>60: continue
                            fx_k,fy_k=K_c[0,0],K_c[1,1]; px_k,py_k=K_c[0,2],K_c[1,2]
                            cx_img=(x1+x2)/2; cy_img=(y1+y2)/2
                            Xc=(cx_img-px_k)*d/fx_k; Yc=(cy_img-py_k)*d/fy_k
                            pt_ego=T_c2e_c@np.array([Xc,Yc,d,1.0])
                            c_e,s_e=math.cos(ego_yaw),math.sin(ego_yaw)
                            wx=ego_x+c_e*pt_ego[0]-s_e*pt_ego[1]
                            wy=ego_y+s_e*pt_ego[0]+c_e*pt_ego[1]
                            vx,vy,spd,hdg=bbox_flow(get_flow(cam,img_c),[x1,y1,x2,y2])
                            score,level=assess_risk_ttc(d,spd,cname)
                            dets_raw.append({'bbox':[x1,y1,x2,y2],'class_name':cname,
                                             'track_id':int(tid),'depth':d,'level':level,'score':score})
                            all_model_dets.append({'cname':cname,'wx':wx,'wy':wy,
                                                   'depth_model':d,'cam':cam})
                except Exception: pass
        else:
            res = yolo(img_c, conf=0.15,
                       classes=list(DETECT_CLASSES.keys()), verbose=False)[0]
            if res.boxes is not None:
                for box in res.boxes:
                    x1,y1,x2,y2=box.xyxy[0].tolist()
                    cid=int(box.cls[0]); cname=DETECT_CLASSES.get(cid,'object')
                    d=bbox_depth(dm,[x1,y1,x2,y2]) if dm is not None else estimate_depth_from_bbox([x1,y1,x2,y2],W_c,H_c)
                    if d<=0 or d>60: continue
                    fx_k,fy_k=K_c[0,0],K_c[1,1]; px_k,py_k=K_c[0,2],K_c[1,2]
                    cx_img=(x1+x2)/2; cy_img=(y1+y2)/2
                    Xc=(cx_img-px_k)*d/fx_k; Yc=(cy_img-py_k)*d/fy_k
                    pt_ego=T_c2e_c@np.array([Xc,Yc,d,1.0])
                    c_e,s_e=math.cos(ego_yaw),math.sin(ego_yaw)
                    wx=ego_x+c_e*pt_ego[0]-s_e*pt_ego[1]
                    wy=ego_y+s_e*pt_ego[0]+c_e*pt_ego[1]
                    score,level=assess_risk_ttc(d,0.0,cname)
                    dets_raw.append({'bbox':[x1,y1,x2,y2],'class_name':cname,
                                     'track_id':-1,'depth':d,'level':level,'score':score})
                    all_model_dets.append({'cname':cname,'wx':wx,'wy':wy,
                                          'depth_model':d,'cam':cam})
        cam_dets[cam] = dets_raw

    # World-space NMS — remove duplicate cross-camera duplicates
    all_model_dets = nms_world(all_model_dets, threshold=3.0)

    # ── Build BEV from model outputs ─────────────────────────────────────────────────
    bev = draw_bev_model(ego_x, ego_y, ego_yaw, ego_hist, all_model_dets)
    # To use GT instead: bev = draw_bev_gt(sample, ego_x, ego_y, ego_yaw, ego_hist, obj_hist, depth_map)

    cv2.putText(bev, f'Frame {f_idx+1}/{MAX_FRAMES}',
                (MAP_SIZE-160,MAP_SIZE-8), cv2.FONT_HERSHEY_SIMPLEX, 0.4,(100,100,100),1)

    # ── Stitch front panorama (3 cameras → 1 seamless image) ──────────────────────────
    front_imgs_raw = []
    for cam in FRONT_CAMS:
        img_c = get_cam_image(sample, cam)
        if img_c is not None:
            ann_img = annotate_camera(img_c, cam_dets.get(cam,[]))
            front_imgs_raw.append(ann_img)
        else:
            front_imgs_raw.append(None)
    front_pano = stitch_panorama(front_imgs_raw)
    front_pano = cv2.cvtColor(front_pano, cv2.COLOR_BGR2RGB)
    front_strip = cv2.resize(front_pano, (PANO_W, TARGET_H))
    cv2.putText(front_strip,'FRONT PANORAMA',(8,22),
                cv2.FONT_HERSHEY_SIMPLEX,0.55,(0,220,220),1)

    # ── Place BEV on black canvas (square, centered) ──────────────────────────────────
    bev_rgb    = cv2.cvtColor(bev, cv2.COLOR_BGR2RGB)
    bev_square = cv2.resize(bev_rgb, (BEV_SQ, BEV_SQ))
    bev_canvas = np.zeros((BEV_SQ, PANO_W, 3), np.uint8)
    x_off = (PANO_W - BEV_SQ) // 2
    bev_canvas[:, x_off:x_off+BEV_SQ, :] = bev_square

    # ── Combine front panorama + BEV into a single frame ───────────────────────────────────────────────────────────────
    divider  = np.full((4, PANO_W, 3), 60, np.uint8)
    combined = np.concatenate([front_strip, divider, bev_canvas], axis=0)
    h,w = combined.shape[:2]
    if h%2: combined=combined[:h-1]
    if w%2: combined=combined[:,:w-1]
    writer.append_data(combined)

writer.close()
print(f'Video saved: {VIDEO_OUTPUT}')


Processing 20 frames...


100%|██████████| 20/20 [04:43<00:00, 14.18s/it]


Video saved: /content/danger_detection_final.mp4


In [ ]:
# ── CELL 17: BEV comparison GT vs Model (single frame) ───────────────────────────────
sample    = samples[10]
ego_x, ego_y, ego_yaw = get_ego_pose(sample)
img_front = get_cam_image(sample, 'CAM_FRONT')
depth_map = run_depth(img_front)

# Render Render GT BEV
bev_gt = draw_bev_gt(sample, ego_x, ego_y, ego_yaw,
                     deque([(ego_x,ego_y)], maxlen=40),
                     defaultdict(lambda: deque(maxlen=20)), depth_map)

# Render Model BEV — using all 6 cameras
depth_maps_dbg = {}
all_dets_dbg   = []
for cam in CAM_NAMES:
    img_c = get_cam_image(sample, cam)
    if img_c is None: continue
    dm = run_depth(img_c)
    depth_maps_dbg[cam] = dm
    K_c=get_intrinsics(sample,cam); T_c2e_c=get_cam2ego(sample,cam)
    H_c,W_c=img_c.shape[:2]
    res=yolo(img_c,conf=0.15,classes=list(DETECT_CLASSES.keys()),verbose=False)[0]
    if res.boxes is None: continue
    for box in res.boxes:
        x1,y1,x2,y2=box.xyxy[0].tolist()
        cid=int(box.cls[0]); cname=DETECT_CLASSES.get(cid,'object')
        d=bbox_depth(dm,[x1,y1,x2,y2])
        if d<=0 or d>60: continue
        fx_k,fy_k=K_c[0,0],K_c[1,1]; px_k,py_k=K_c[0,2],K_c[1,2]
        cx_img=(x1+x2)/2; cy_img=(y1+y2)/2
        Xc=(cx_img-px_k)*d/fx_k; Yc=(cy_img-py_k)*d/fy_k
        pt_ego=T_c2e_c@np.array([Xc,Yc,d,1.0])
        c_e,s_e=math.cos(ego_yaw),math.sin(ego_yaw)
        wx=ego_x+c_e*pt_ego[0]-s_e*pt_ego[1]
        wy=ego_y+s_e*pt_ego[0]+c_e*pt_ego[1]
        all_dets_dbg.append({'cname':cname,'wx':wx,'wy':wy,'depth_model':d,'cam':cam})

all_dets_dbg = nms_world(all_dets_dbg, threshold=3.0)
bev_model = draw_bev_model(ego_x, ego_y, ego_yaw,
                           deque([(ego_x,ego_y)], maxlen=40), all_dets_dbg)

# Front panorama
pano_imgs = [get_cam_image(sample, cam) for cam in FRONT_CAMS]
pano = stitch_panorama(pano_imgs)

fig, axes = plt.subplots(1,3,figsize=(22,7))
axes[0].imshow(cv2.cvtColor(pano, cv2.COLOR_BGR2RGB))
axes[0].set_title('Front Panorama'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(bev_gt,cv2.COLOR_BGR2RGB))
axes[1].set_title('BEV — nuScenes GT'); axes[1].axis('off')
axes[2].imshow(cv2.cvtColor(bev_model,cv2.COLOR_BGR2RGB))
axes[2].set_title(f'BEV — YOLO11+DA2 ({len(all_dets_dbg)} objects)'); axes[2].axis('off')
plt.tight_layout()
plt.savefig('/content/bev_compare.png',dpi=120,bbox_inches='tight')
plt.show()
print(f'GT objects: {len([a for a in sample["anns"]])}  |  Model detections: {len(all_dets_dbg)}')


GT objects: 127  |  Model detections: 25


In [ ]:
# ── CELL 18: EVALUATION — Full comparison: GT vs Model ───────────────────────────
EVAL_FRAMES = min(10, len(samples))
DIST_THRESH = 5.0

total_gt=0; total_det=0; total_tp=0
depth_errors=[]; missed_cats={}; fp_count=0

for f_idx in range(EVAL_FRAMES):
    sample    = samples[f_idx]
    ego_x,ego_y,ego_yaw = get_ego_pose(sample)
    img_front = get_cam_image(sample,'CAM_FRONT')
    dm_front  = run_depth(img_front) if img_front is not None else None

    gt_objects = []
    for ann_token in sample['anns']:
        ann = nusc.get('sample_annotation',ann_token)
        cat = ann['category_name']
        if any(cat.startswith(s) for s in SKIP_CATS): continue
        pos  = ann['translation']
        dist = math.hypot(pos[0]-ego_x, pos[1]-ego_y)
        if dist > MAP_RANGE: continue
        sd=nusc.get('sample_data',sample['data']['CAM_FRONT'])
        ep=nusc.get('ego_pose',sd['ego_pose_token'])
        R_e=Quaternion(ep['rotation']).rotation_matrix
        t_e=np.array(ep['translation'])
        T_e2w=np.eye(4); T_e2w[:3,:3]=R_e; T_e2w[:3,3]=t_e
        T_c2e_f=get_cam2ego(sample,'CAM_FRONT')
        pt_e=np.linalg.inv(T_e2w)@np.append(np.array(pos[:3]),1.0)
        pt_c=np.linalg.inv(T_c2e_f)@pt_e
        gt_depth=float(pt_c[2]) if pt_c[2]>0 else None
        gt_objects.append({'cat':cat,'wx':pos[0],'wy':pos[1],'dist':dist,'gt_depth':gt_depth})
    total_gt += len(gt_objects)

    model_objects=[]
    for cam in CAM_NAMES:
        img_c=get_cam_image(sample,cam)
        if img_c is None: continue
        dm=dm_front if cam=='CAM_FRONT' else None
        K_c=get_intrinsics(sample,cam); T_c2e_c=get_cam2ego(sample,cam)
        H_c,W_c=img_c.shape[:2]
        res=yolo(img_c,conf=0.15,classes=list(DETECT_CLASSES.keys()),verbose=False)[0]
        if res.boxes is None: continue
        for box in res.boxes:
            x1,y1,x2,y2=box.xyxy[0].tolist()
            cid=int(box.cls[0]); cname=DETECT_CLASSES.get(cid,'object')
            d=bbox_depth(dm,[x1,y1,x2,y2]) if dm is not None else estimate_depth_from_bbox([x1,y1,x2,y2],W_c,H_c)
            if d<=0 or d>60: continue
            fx_k,fy_k=K_c[0,0],K_c[1,1]; px_k,py_k=K_c[0,2],K_c[1,2]
            cx_img=(x1+x2)/2; cy_img=(y1+y2)/2
            Xc=(cx_img-px_k)*d/fx_k; Yc=(cy_img-py_k)*d/fy_k
            pt_ego=T_c2e_c@np.array([Xc,Yc,d,1.0])
            c_e,s_e=math.cos(ego_yaw),math.sin(ego_yaw)
            wx=ego_x+c_e*pt_ego[0]-s_e*pt_ego[1]
            wy=ego_y+s_e*pt_ego[0]+c_e*pt_ego[1]
            model_objects.append({'cname':cname,'wx':wx,'wy':wy,'depth_model':d,'cam':cam})
    model_objects=nms_world(model_objects,threshold=3.0)
    total_det+=len(model_objects)

    matched=set()
    for gt in gt_objects:
        best_dist=DIST_THRESH; best_idx=-1
        for j,mo in enumerate(model_objects):
            if j in matched: continue
            d_match=math.hypot(gt['wx']-mo['wx'],gt['wy']-mo['wy'])
            if d_match<best_dist: best_dist=d_match; best_idx=j
        if best_idx>=0:
            total_tp+=1; matched.add(best_idx)
            mo=model_objects[best_idx]
            if gt['gt_depth'] is not None and mo['cam']=='CAM_FRONT':
                err=abs(mo['depth_model']-gt['gt_depth'])
                depth_errors.append({'gt':gt['gt_depth'],'model':mo['depth_model'],'err':err,'cat':gt['cat']})
        else:
            cat_s=gt['cat'].split('.')[-1]
            missed_cats[cat_s]=missed_cats.get(cat_s,0)+1
    fp_count+=len(model_objects)-len(matched)

precision=total_tp/max(total_det,1)
recall=total_tp/max(total_gt,1)
f1=2*precision*recall/max(precision+recall,1e-6)

print('='*60)
print('EVALUATION: nuScenes GT vs YOLO11 + Depth Anything V2')
print('='*60)
print(f'GT objects:     {total_gt}')
print(f'Model dets:     {total_det}')
print(f'True Positives: {total_tp}')
print(f'False Positives:{fp_count}')
print(f'False Negatives:{total_gt-total_tp}')
print(f'Precision:      {precision:.2%}')
print(f'Recall:         {recall:.2%}')
print(f'F1 Score:       {f1:.2%}')
if depth_errors:
    errs=[e['err'] for e in depth_errors]
    print(f'Depth MAE:      {np.mean(errs):.2f} m')
    print(f'Depth RMSE:     {np.sqrt(np.mean([e**2 for e in errs])):.2f} m')
print('\nMissed by category:')
for cat,cnt in sorted(missed_cats.items(),key=lambda x:-x[1])[:8]:
    print(f'  {cat:20s}  {cnt}')


EVALUATION: nuScenes GT vs YOLO11 + Depth Anything V2
GT objects:     133
Model dets:     224
True Positives: 16
False Positives:208
False Negatives:117
Precision:      7.14%
Recall:         12.03%
F1 Score:       8.96%
Depth MAE:      3.40 m
Depth RMSE:     3.50 m

Missed by category:
  adult                 92
  car                   12
  pushable_pullable     10
  truck                 3


In [ ]:
# ── CELL 19: Preview + Download ─────────────────────────────────────────────
from IPython.display import HTML, display
from base64 import b64encode
from google.colab import files

with open(VIDEO_OUTPUT,'rb') as f:
    data = b64encode(f.read()).decode()

display(HTML(f'''
<div style="text-align:center">
  <h3>Danger Detection — Final Output</h3>
  <video width="900" controls autoplay loop>
    <source src="data:video/mp4;base64,{data}" type="video/mp4">
  </video>
</div>
'''))

files.download(VIDEO_OUTPUT)


Output hidden; open in https://colab.research.google.com to view.